# YOUR PROJECT TITLE

> **Note the following:** 
> 1. This is *not* meant to be an example of an actual **data analysis project**, just an example of how to structure such a project.
> 1. Remember the general advice on structuring and commenting your code
> 1. The `dataproject.py` file includes a function which can be used multiple times in this notebook.

In [1]:
# The DST API wrapper
%pip install git+https://github.com/alemartinello/dstapi

  Cloning https://github.com/alemartinello/dstapi to /private/var/folders/lb/2stvshn97wz4m8c3hcpvz9bh0000gn/T/pip-req-build-f522xv5q
  Running command git clone --filter=blob:none --quiet https://github.com/alemartinello/dstapi /private/var/folders/lb/2stvshn97wz4m8c3hcpvz9bh0000gn/T/pip-req-build-f522xv5q
  Resolved https://github.com/alemartinello/dstapi to commit d9eeb5a82cbc70b7d63b2ff44d92632fd77123a4
  Preparing metadata (setup.py) ... done
Note: you may need to restart the kernel to use updated packages.


Imports and set magics:

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
#from matplotlib_venn import venn2
import re

# autoreload modules when code is run
%load_ext autoreload
%autoreload 2

# user written modules
import dataproject

from dstapi import DstApi


# Read and clean data

Importing data, through an API and loading it:

In [3]:
proj=DstApi('FRKM123')

A quick overview of the available data

In [4]:
tabsum = proj.tablesummary(language='en')
display(tabsum)

Table FRKM123: Population projections 2023 by region, age, sex and time
Last update: 2023-06-01T08:00:00


,variable name,# values,First value,First value label,Last value,Last value label,Time variable
0,OMRÅDE,99,101,Copenhagen,851,Aalborg,False
1,ALDER,102,TOT,"Age, total",100-,100 years and over,False
2,KØN,2,M,Men,K,Women,False
3,Tid,28,2023,2023,2050,2050,True


***Clean data***

In [5]:
params = proj._define_base_params(language='en')
params

{'table': 'frkm123',
 'format': 'BULK',
 'lang': 'en',
 'variables': [{'code': 'OMRÅDE', 'values': ['*']},
  {'code': 'ALDER', 'values': ['*']},
  {'code': 'KØN', 'values': ['*']},
  {'code': 'Tid', 'values': ['*']}]}

In [6]:
proj_api=proj.get_data(params=params)
proj_api.head(5)

,OMRÅDE,ALDER,KØN,TID,INDHOLD
0,Fredensborg,19 years,Women,2034,223
1,Fredensborg,19 years,Men,2034,240
2,Fredensborg,2 years,Women,2034,240
3,Fredensborg,2 years,Men,2034,268
4,Fredensborg,20 years,Women,2034,172


Rename variables

In [7]:
proj_api.rename(columns = {'OMRÅDE':'municipality'}, inplace=True)
proj_api.rename(columns = {'ALDER':'age'}, inplace=True)
proj_api.rename(columns = {'KØN':'sex'}, inplace=True)
proj_api.rename(columns = {'TID':'year'}, inplace=True)
proj_api.rename(columns = {'INDHOLD':'population_projection'}, inplace=True)
proj_api.head(5)

,municipality,age,sex,year,population_projection
0,Fredensborg,19 years,Women,2034,223
1,Fredensborg,19 years,Men,2034,240
2,Fredensborg,2 years,Women,2034,240
3,Fredensborg,2 years,Men,2034,268
4,Fredensborg,20 years,Women,2034,172


In [8]:
df=pd.DataFrame(proj_api)
df['pop_proj_all'] = ""

df.head(5)

,municipality,age,sex,year,population_projection,pop_proj_all
0,Fredensborg,19 years,Women,2034,223,
1,Fredensborg,19 years,Men,2034,240,
2,Fredensborg,2 years,Women,2034,240,
3,Fredensborg,2 years,Men,2034,268,
4,Fredensborg,20 years,Women,2034,172,


Summarize values of population_projection if municipality equals each other, age equals each other and year equals each other.

In [9]:
def sumval(group):
    if len(group)>1:
        return group.sum()
    else:
        return group.values[0]
    
df['pop_proj_all']=df.groupby(['municipality','age','year'])['population_projection'].transform(sumval)
df.head(5)

,municipality,age,sex,year,population_projection,pop_proj_all
0,Fredensborg,19 years,Women,2034,223,463
1,Fredensborg,19 years,Men,2034,240,463
2,Fredensborg,2 years,Women,2034,240,508
3,Fredensborg,2 years,Men,2034,268,508
4,Fredensborg,20 years,Women,2034,172,371


In [10]:
df=df[df['sex']!='Men']
df.head(5)

,municipality,age,sex,year,population_projection,pop_proj_all
0,Fredensborg,19 years,Women,2034,223,463
2,Fredensborg,2 years,Women,2034,240,508
4,Fredensborg,20 years,Women,2034,172,371
6,Fredensborg,21 years,Women,2034,135,290
8,Fredensborg,22 years,Women,2034,106,237


In [11]:
columns_to_drop=['sex','population_projection']
df=df.drop(columns_to_drop, axis=1)

In [12]:
df.head(5)

,municipality,age,year,pop_proj_all
0,Fredensborg,19 years,2034,463
2,Fredensborg,2 years,2034,508
4,Fredensborg,20 years,2034,371
6,Fredensborg,21 years,2034,290
8,Fredensborg,22 years,2034,237


In [13]:
muni_type=df['municipality'].dtype
age_type=df['age'].dtype
year_type=df['year'].dtype
pop_type=df['pop_proj_all'].dtype
print(muni_type)
print(age_type)
print(year_type)
print(pop_type)

object
object
int64
int64


In [14]:
print(type('age'))

<class 'str'>


In [15]:
df=df[df['age']!='Age, total']

In [16]:
df['age'] = df['age'].str.split(" y").str[0].astype(int)
df.head(5)

,municipality,age,year,pop_proj_all
0,Fredensborg,19,2034,463
2,Fredensborg,2,2034,508
4,Fredensborg,20,2034,371
6,Fredensborg,21,2034,290
8,Fredensborg,22,2034,237


In [17]:
# Define age ranges
age_bins = range(df['age'].min(), df['age'].max() + 11, 10)

# Create labels for the age ranges
labels = [f'{i}-{i+9}' if i != 100 else f'{i}+' for i in age_bins[:-1]]

# Use pd.cut() to sort ages into age ranges
df['age_group'] = pd.cut(df['age'], bins=age_bins, labels=labels, right=False)

# Function to summarize values in 'Value' column for each group
def sumval(group):
    if len(group) > 1:
        return group.sum()
    else:
        return group.values[0]

# Applying the function to create the new column
df['pop_proj_age'] = df.groupby(['municipality', 'age_group', 'year'])['pop_proj_all'].transform(sumval)

print(df)

/var/folders/lb/2stvshn97wz4m8c3hcpvz9bh0000gn/T/ipykernel_74308/887293629.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df['pop_proj_age'] = df.groupby(['municipality', 'age_group', 'year'])['pop_proj_all'].transform(sumval)


       municipality  age  year  pop_proj_all age_group  pop_proj_age
0       Fredensborg   19  2034           463     10-19          5270
2       Fredensborg    2  2034           508       0-9          5281
4       Fredensborg   20  2034           371     20-29          2687
6       Fredensborg   21  2034           290     20-29          2687
8       Fredensborg   22  2034           237     20-29          2687
...             ...  ...   ...           ...       ...           ...
565478   Jammerbugt   42  2050           418     40-49          4248
565480   Jammerbugt   43  2050           429     40-49          4248
565482   Jammerbugt   44  2050           425     40-49          4248
565484   Jammerbugt   45  2050           427     40-49          4248
565486   Jammerbugt   46  2050           427     40-49          4248

[279972 rows x 6 columns]


In [25]:
df = df.sort_values(by=['municipality','year','age'])
df.head(5)

,municipality,age,year,pop_proj_all,age_group,pop_proj_age
136402,Aabenraa,0,2023,501,0-9,5820
136404,Aabenraa,1,2023,547,0-9,5820
136428,Aabenraa,2,2023,540,0-9,5820
136450,Aabenraa,3,2023,592,0-9,5820
136472,Aabenraa,4,2023,637,0-9,5820
136494,Aabenraa,5,2023,597,0-9,5820
136516,Aabenraa,6,2023,647,0-9,5820
136538,Aabenraa,7,2023,545,0-9,5820
136560,Aabenraa,8,2023,604,0-9,5820
136582,Aabenraa,9,2023,610,0-9,5820


In [26]:
df_unique = df.drop_duplicates(subset=['municipality', 'year', 'pop_proj_age'])
df_unique.head(5)

,municipality,age,year,pop_proj_all,age_group,pop_proj_age
136402,Aabenraa,0,2023,501,0-9,5820
136406,Aabenraa,10,2023,623,10-19,7129
136430,Aabenraa,20,2023,677,20-29,5297
136452,Aabenraa,30,2023,629,30-39,6192
136474,Aabenraa,40,2023,605,40-49,7020
136496,Aabenraa,50,2023,780,50-59,8744
136518,Aabenraa,60,2023,830,60-69,8082
136540,Aabenraa,70,2023,772,70-79,7026
136562,Aabenraa,80,2023,527,80-89,3136
136584,Aabenraa,90,2023,134,90-99,550


In [31]:
columns_to_drop_2=['age','pop_proj_all']
df=df_unique.drop(columns_to_drop_2, axis=1)


In [32]:
df.head(50)

,municipality,year,age_group,pop_proj_age
136402,Aabenraa,2023,0-9,5820
136406,Aabenraa,2023,10-19,7129
136430,Aabenraa,2023,20-29,5297
136452,Aabenraa,2023,30-39,6192
136474,Aabenraa,2023,40-49,7020
136496,Aabenraa,2023,50-59,8744
136518,Aabenraa,2023,60-69,8082
136540,Aabenraa,2023,70-79,7026
136562,Aabenraa,2023,80-89,3136
136584,Aabenraa,2023,90-99,550


In [35]:
#Resetting the index:
df.reset_index(inplace = True, drop = True) # Drop old index too
df.rename(columns = {'pop_proj_age':'population_projection'}, inplace=True)
df.head(5)

,municipality,year,age_group,population_projection
0,Aabenraa,2023,0-9,5820
1,Aabenraa,2023,10-19,7129
2,Aabenraa,2023,20-29,5297
3,Aabenraa,2023,30-39,6192
4,Aabenraa,2023,40-49,7020


## Explore each data set

In order to be able to **explore the raw data**, you may provide **static** and **interactive plots** to show important developments 

**Interactive plot** :

In [20]:
def plot_func():
    # Function that operates on data set
    pass

widgets.interact(plot_func, 
    # Let the widget interact with data through plot_func()    
); 


interactive(children=(Output(),), _dom_classes=('widget-interact',))

Explain what you see when moving elements of the interactive plot around. 

# Merge data sets

Now you create combinations of your loaded data sets. Remember the illustration of a (inner) **merge**:

In [21]:
plt.figure(figsize=(15,7))
v = venn2(subsets = (4, 4, 10), set_labels = ('Data X', 'Data Y'))
v.get_label_by_id('100').set_text('dropped')
v.get_label_by_id('010').set_text('dropped' )
v.get_label_by_id('110').set_text('included')
plt.show()

NameError: name 'venn2' is not defined

<Figure size 1500x700 with 0 Axes>

Here we are dropping elements from both data set X and data set Y. A left join would keep all observations in data X intact and subset only from Y. 

Make sure that your resulting data sets have the correct number of rows and columns. That is, be clear about which observations are thrown away. 

**Note:** Don't make Venn diagrams in your own data project. It is just for exposition. 

# Analysis

To get a quick overview of the data, we show some **summary statistics** on a meaningful aggregation. 

MAKE FURTHER ANALYSIS. EXPLAIN THE CODE BRIEFLY AND SUMMARIZE THE RESULTS.

# Conclusion

ADD CONCISE CONLUSION.